# Fase 2 — Limpeza, Vazamento de Dados e Classificadores

Continuação a partir de `../dados/fakerecogna_bruto.csv`.

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib as mpl
import matplotlib.pyplot as plt

CORAL = "#EF476F"   # Fake
AZUL = "#118AB2"    # Real

mpl.rcParams.update({
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#073B4C",
    "axes.labelsize": 10.5,
    "axes.labelweight": "bold",
    "axes.edgecolor": "#999999",
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#E6E6E6",
    "grid.linewidth": 0.7,
    "legend.frameon": False,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})

df = pd.read_csv("../dados/fakerecogna_bruto.csv")
print(f"{df.shape[0]} linhas carregadas.")

## Limpeza

In [ ]:
df = df.dropna(subset=["Noticia", "Classe", "Categoria", "URL"]).reset_index(drop=True)
df["Classe"] = df["Classe"].astype(int)
print(f"{len(df)} linhas após remoção de registros incompletos.")

## Verificação de vazamento — coluna Subtítulo

In [ ]:
df["tem_subtitulo"] = df["Subtitulo"].notna()

tabela_leak = pd.crosstab(df["tem_subtitulo"], df["Classe"], normalize="columns") * 100
tabela_leak.index = ["SEM SUBTÍTULO", "COM SUBTÍTULO"]
tabela_leak.columns = ["FAKE", "REAL"]
print(tabela_leak.round(1))

fig, ax = plt.subplots(figsize=(6, 4.2))
tabela_leak.plot(kind="bar", ax=ax, color=[CORAL, AZUL], width=0.6)
ax.set_title("PRESENÇA DO SUBTÍTULO POR CLASSE", loc="left")
ax.set_ylabel("% DENTRO DE CADA CLASSE")
plt.xticks(rotation=0)
ax.legend(title="CLASSE")
plt.savefig("../figuras/verificacao_leakage_subtitulo.png")
plt.show()

**Decisão:** `Subtitulo`/`tem_subtitulo` excluídos das features — presença desigual entre classes indica vazamento editorial, não conteúdo discriminativo.

## Tratamento da coluna Data

In [ ]:
def extrair_data(texto):
    if pd.isna(texto):
        return pd.NaT
    texto = str(texto)
    for padrao in [r"(\d{1,2})[/-](\d{1,2})[/-](\d{4})", r"(\d{4})[/-](\d{1,2})[/-](\d{1,2})"]:
        m = re.search(padrao, texto)
        if m:
            g = m.groups()
            ano, mes, dia = (g[2], g[1], g[0]) if len(g[0]) == 2 else (g[0], g[1], g[2])
            try:
                return pd.Timestamp(year=int(ano), month=int(mes), day=int(dia))
            except (ValueError, pd.errors.OutOfBoundsDatetime):
                return pd.NaT
    return pd.NaT

df["data_tratada"] = df["Data"].apply(extrair_data)
print(f"Datas convertidas: {df['data_tratada'].notna().sum()}/{len(df)}")

## Divisão treino/teste e vetorização TF-IDF

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

X = df["Noticia"]
y = df["Classe"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vetorizador = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words=stopwords.words("portuguese"), min_df=5)
X_train_tfidf = vetorizador.fit_transform(X_train)
X_test_tfidf = vetorizador.transform(X_test)
print(f"Treino: {X_train_tfidf.shape} | Teste: {X_test_tfidf.shape}")

## Treinamento: Logistic Regression e Random Forest

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

modelo_lr = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_tfidf, y_train)
modelo_rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_train_tfidf, y_train)
print("Modelos treinados.")

## Avaliação

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from matplotlib.colors import LinearSegmentedColormap

for nome, modelo in [("Logistic Regression", modelo_lr), ("Random Forest", modelo_rf)]:
    pred = modelo.predict(X_test_tfidf)
    print(f"\n{nome} — Acurácia: {accuracy_score(y_test, pred):.3f}")
    print(classification_report(y_test, pred, target_names=["Fake (0)", "Real (1)"]))

In [ ]:
cmap_vivido = LinearSegmentedColormap.from_list("vivido", ["#FFFFFF", AZUL])

fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))
for ax, (nome, modelo) in zip(axes, [("LOGISTIC REGRESSION", modelo_lr), ("RANDOM FOREST", modelo_rf)]):
    pred = modelo.predict(X_test_tfidf)
    cm = confusion_matrix(y_test, pred)
    ax.imshow(cm, cmap=cmap_vivido, vmin=0, vmax=cm.max())
    for i in range(2):
        for j in range(2):
            cor = "white" if cm[i, j] > cm.max() * 0.55 else "#111111"
            ax.text(j, i, f"{cm[i, j]:,}".replace(",", "."), ha="center", va="center", fontsize=13, color=cor, fontweight="bold")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["FAKE", "REAL"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["FAKE", "REAL"])
    ax.set_xlabel("PREVISTO"); ax.set_ylabel("REAL")
    ax.set_title(nome, loc="left")
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)
plt.tight_layout()
plt.savefig("../figuras/matrizes_confusao.png")
plt.show()

## Termos mais relevantes (Logistic Regression)

In [ ]:
nomes_features = np.array(vetorizador.get_feature_names_out())
coef = modelo_lr.coef_[0]
print("FAKE:", ", ".join(nomes_features[np.argsort(coef)[:15]]))
print("REAL:", ", ".join(nomes_features[np.argsort(coef)[-15:][::-1]]))

## Exportação dos resultados

In [ ]:
indices_teste = X_test.index
resultados = pd.DataFrame({
    "titulo": df.loc[indices_teste, "Titulo"].values,
    "categoria": df.loc[indices_teste, "Categoria"].values,
    "data": df.loc[indices_teste, "data_tratada"].values,
    "n_palavras": df.loc[indices_teste, "n_palavras_noticia"].values,
    "classe_real": y_test.values,
    "previsao_regressao_logistica": modelo_lr.predict(X_test_tfidf),
    "previsao_random_forest": modelo_rf.predict(X_test_tfidf),
})
resultados["acertou_lr"] = resultados["classe_real"] == resultados["previsao_regressao_logistica"]
resultados["acertou_rf"] = resultados["classe_real"] == resultados["previsao_random_forest"]
resultados["classe_real_label"] = resultados["classe_real"].map({0: "Fake", 1: "Real"})

resultados.to_csv("../dados/resultados_classificacao.csv", index=False, encoding="utf-8-sig")
print(f"Exportado: {len(resultados)} linhas")